# 08 — Screening des paires (univers canonique S&P 500 + 8 momentum = 215 titres)

Ce notebook **remplace** `08a`, `08a_bis` et `08b`. Il est un simple appelant de
`src/screening.py` (logique factorisée). Il produit trois artefacts committés :

- `data/sp500full_pairs_ranked.csv` — paires cointégrées (EG + Benjamini-Hochberg), classées par SCORE
- `data/sp500full_all_trials_theta.csv` — le θ (Sharpe par pari, ch. 15 López de Prado) de **chaque** candidate → « record every backtest »
- le **Deflated Sharpe Ratio** du gagnant, déflaté sur le vrai nombre d'essais (ch. 14)

Univers canonique : les 215 titres (`sp500full`), qui incluent les 8 titres momentum.

In [1]:
import sys; sys.path.insert(0, "../src")
import pandas as pd, numpy as np
import screening as S

adj = pd.read_csv("../data/sp500full_adj_close.csv", index_col=0, parse_dates=True)
sectors = pd.read_csv("../data/sp500full_sectors.csv", index_col="ticker")["secteur"]
years = len(adj) / 252
print("Univers :", adj.shape[1], "titres |", adj.index.min().date(), "->", adj.index.max().date())

Univers : 215 titres | 2011-11-17 -> 2026-08-12


## 1. Screening de cointégration (Engle-Granger + Benjamini-Hochberg + SCORE)

`screen_cointegration` reconstitue les candidates intra-secteur (corrélation ≥ 0.40),
teste la cointégration dans les deux directions (corrige l'asymétrie OLS), applique
la correction de tests multiples BH, puis score les survivantes.

In [2]:
ranked, n_cand = S.screen_cointegration(adj, sectors)
ranked.to_csv("../data/sp500full_pairs_ranked.csv", index=False)
print(f"Candidates testées : {n_cand} | cointégrées après BH : {len(ranked)}")
ranked[["paire","secteur","p_bh","half_life","subwin_frac","beta_cv","SCORE"]].round(3)

Candidates testées : 1006 | cointégrées après BH : 7


,paire,secteur,p_bh,half_life,subwin_frac,beta_cv,SCORE
0,BAC/PNC,Banks,0.020,46.746,0.250,0.356,0.716
1,ADI/MPWR,Semis,0.006,43.190,0.250,0.822,0.683
2,ICE/MCO,CapitalMarkets,0.019,49.426,0.125,0.608,0.656
3,MCD/YUM,Food_Restaurant,0.035,51.501,0.125,0.803,0.638
4,AMAT/AMD,Semis,0.000,43.293,0.000,1.556,0.578
5,AMAT/MPWR,Semis,0.006,96.501,0.125,0.737,0.551
6,ED/SRE,Utilities,0.006,68.499,0.000,0.780,0.511


## 2. Record every backtest + Deflated Sharpe Ratio (López de Prado ch. 11 / 14 / 15)

On rejoue **toutes** les candidates et on enregistre le θ de chacune (Sharpe par pari
annualisé, robuste aux périodes à plat). Puis on déflate le Sharpe du gagnant sur cette
distribution : SR\* est le maximum attendu sous H₀ (vrai Sharpe nul) après N essais.

In [3]:
trials = S.record_all_trials(adj, sectors)
trials.to_csv("../data/sp500full_all_trials_theta.csv", index=False)

# gagnant par Sharpe réel parmi les survivantes cointégrées
best, best_bets, best_theta = None, None, -1e9
for p in ranked["paire"]:
    a, b = p.split("/")
    _, bets = S.vec_pair_backtest(adj, a, b)
    th, *_ = S.bet_theta(bets, years)
    if not np.isnan(th) and th > best_theta:
        best, best_bets, best_theta = p, bets, th

out = S.deflated_sharpe(trials["theta"], best_bets, years)
print(f"Essais enregistrés (N)        : {out['N']}")
print(f"Gagnant par Sharpe réel       : {best}  (θ = {out['theta_winner']:.3f}, rang {out['rank']}/{out['N']})")
print(f"SR* attendu sous H0           : {out['sr_star']:.3f}")
print(f"--> Deflated Sharpe Ratio     : {out['dsr']:.3f}")

Essais enregistrés (N)        : 1006
Gagnant par Sharpe réel       : ADI/MPWR  (θ = 0.667, rang 77/1006)
SR* attendu sous H0           : 1.052
--> Deflated Sharpe Ratio     : 0.097


## 3. Lecture

Le DSR est **très en dessous de 0.95** : déflatée sur ~1000 essais, la meilleure paire
ne se distingue pas d'un tirage chanceux. Ce n'est pas un défaut de rentabilité mais un
**manque de puissance** (échantillon court, peu de paris). Conséquence assumée au rapport :
le sleeve paires est présenté comme **contribution de diversification**, pas comme alpha
autonome. Piste constructive : trader un **panier** des meilleures paires plutôt qu'une seule.